In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import os

# Configuración de dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


In [2]:
# --- 1. Definición de la Arquitectura (Ajustada a tu .pth) ---
class SimpleCNN2Digits(nn.Module):
    def __init__(self):
        super(SimpleCNN2Digits, self).__init__()
        # Arquitectura de 16 y 32 filtros (la que encaja con tu archivo)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Entrada: 32 canales * 7 alto * 14 ancho = 3136
        self.fc1 = nn.Linear(32 * 7 * 14, 128)
        self.fc2 = nn.Linear(128, 100) # 100 clases (00-99)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 14) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- 2. Carga del Modelo ---
model_path = "../models/simple_cnn_2digits.pth"
model = SimpleCNN2Digits().to(device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("✅ Modelo cargado y listo para inferencia.")
else:
    print(f"❌ Error: No se encontró el modelo en {model_path}")

# --- 3. Carga de los Datasets (Las tres piezas) ---
data_dir = "../data"

try:
    # A. El dataset tabular (Ligero)
    df_tabular = pd.read_csv(os.path.join(data_dir, "m_tabular.csv"))
    
    # B. El dataset de imágenes de clientes (Pickle)
    df_cust_img = pd.read_pickle(os.path.join(data_dir, "m_cust_images.pkl"))
    
    # C. El dataset de imágenes de vendedores (Pickle)
    df_sell_img = pd.read_pickle(os.path.join(data_dir, "m_sell_images.pkl"))
    
    print("\n✅ Datasets cargados exitosamente:")
    print(f"- Tabular: {df_tabular.shape}")
    print(f"- Imágenes Cliente: {df_cust_img.shape}")
    print(f"- Imágenes Vendedor: {df_sell_img.shape}")

except FileNotFoundError as e:
    print(f"❌ Error al cargar los datos: {e}")

# Verificación rápida del contenido
display(df_cust_img.head(2))

✅ Modelo cargado y listo para inferencia.

✅ Datasets cargados exitosamente:
- Tabular: (119143, 40)
- Imágenes Cliente: (119143, 2)
- Imágenes Vendedor: (119143, 2)


,order_id,customer_state_image
0,e481f51cbdc54678b7cc49136f2d6af7,"[[[tensor(0.), tensor(0.), tensor(0.), tensor(..."
1,e481f51cbdc54678b7cc49136f2d6af7,"[[[tensor(0.), tensor(0.), tensor(0.), tensor(..."


In [3]:
# --- CELDA 1: Inferencia Clientes ---
print("🔍 Procesando imágenes de Clientes...")
model.eval()
customer_preds = []

with torch.no_grad():
    for img in df_cust_img['customer_state_image']:
        if torch.is_tensor(img):
            # Preparar batch [1, 1, 28, 56] y enviar a dispositivo
            input_tensor = img.unsqueeze(0).to(device)
            output = model(input_tensor)
            pred = torch.argmax(output, dim=1).item()
            customer_preds.append(pred)
        else:
            customer_preds.append(-1)

# Asignar predicciones y limpiar tensores
df_cust_img['customer_state_num_pred'] = customer_preds
df_cust_img.drop(columns=['customer_state_image'], inplace=True)

print(f"✅ Inferencia de clientes finalizada. Predicciones generadas: {len(customer_preds)}")
display(df_cust_img.head(3))

🔍 Procesando imágenes de Clientes...
✅ Inferencia de clientes finalizada. Predicciones generadas: 119143


,order_id,customer_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25
1,e481f51cbdc54678b7cc49136f2d6af7,25
2,e481f51cbdc54678b7cc49136f2d6af7,25


In [4]:
# --- CELDA 2: Inferencia Vendedores ---
print("🔍 Procesando imágenes de Vendedores...")
seller_preds = []

with torch.no_grad():
    for img in df_sell_img['seller_state_image']:
        if torch.is_tensor(img):
            input_tensor = img.unsqueeze(0).to(device)
            output = model(input_tensor)
            pred = torch.argmax(output, dim=1).item()
            seller_preds.append(pred)
        else:
            seller_preds.append(-1)

# Asignar predicciones y limpiar tensores
df_sell_img['seller_state_num_pred'] = seller_preds
df_sell_img.drop(columns=['seller_state_image'], inplace=True)

print(f"✅ Inferencia de vendedores finalizada. Predicciones generadas: {len(seller_preds)}")
display(df_sell_img.head(3))

🔍 Procesando imágenes de Vendedores...
✅ Inferencia de vendedores finalizada. Predicciones generadas: 119143


,order_id,seller_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25
1,e481f51cbdc54678b7cc49136f2d6af7,25
2,e481f51cbdc54678b7cc49136f2d6af7,25


In [5]:
# --- CELDA 3: Unificación Final ---
print("🔗 Unificando datasets...")

# 1. Unir el tabular con las predicciones del cliente
m_final = pd.merge(df_tabular, df_cust_img, on='order_id', how='left')

# 2. Unir el resultado con las predicciones del vendedor
m_final = pd.merge(m_final, df_sell_img, on='order_id', how='left')

# 3. Verificación de integridad
# Revisamos si las predicciones de la IA coinciden con los números originales (si existían)
check_cols = ['order_id', 'customer_state_num', 'customer_state_num_pred', 'seller_state_num', 'seller_state_num_pred']

print("✅ Unificación completada exitosamente.")
print(f"Dimensiones finales del dataset: {m_final.shape}")

# Mostrar una comparativa para validar la precisión de la IA
display(m_final[check_cols].head(10))

# Opcional: Guardar el dataset final limpio para el entrenamiento del modelo de retrasos
# m_final.to_csv("../data/olist_integrated_with_ai.csv", index=False)

🔗 Unificando datasets...
✅ Unificación completada exitosamente.
Dimensiones finales del dataset: (1014553, 42)


,order_id,customer_state_num,customer_state_num_pred,seller_state_num,seller_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
1,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
2,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
3,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
4,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
5,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
6,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
7,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
8,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25
9,e481f51cbdc54678b7cc49136f2d6af7,25,25,25,25


In [20]:
print(m_final["customer_state_num"].unique())

[25  4  8 19 17 22 18 10 23 21 15 26  5  6 24 12 14 13 20  7  3 11  9 16
  1  0  2]


In [17]:
m_final["customer_state_num_pred"].unique()

array([25,  4,  8, 19, 17, 22, 18, 10, 23, 21, 15, 26,  5,  6, 24, 12, 14,
       13, 20,  7,  3, 11,  9, 16,  1,  0,  2])

In [18]:
m_final["seller_state_num"].unique()

array([25, 10,  7, 22,  6, 17, 23, 18,  8,  4, -1,  9,  0, 14, 15,  5, 12,
       16, 19, 11, 13,  2, 24, 20])

In [19]:
m_final["seller_state_num_pred"].unique()

array([25, 10,  7, 22,  6, 17, 23, 18,  8,  4, 32,  9,  0, 14, 15,  5, 12,
       16, 19, 11, 13,  2, 24, 20])

In [ ]:
# # Convertir todas las columnas de tiempo a datetime
# date_cols = [col for col in m.columns if 'timestamp' in col or 'date' in col]
# for col in date_cols:
#     m[col] = pd.to_datetime(m[col])

# # CREACIÓN DE LA VARIABLE OBJETIVO (Target)
# # Un pedido está retrasado si la fecha de entrega real es mayor a la estimada.
# # Solo podemos calcular esto para pedidos que ya fueron entregados.

# # Filtramos solo pedidos entregados para el entrenamiento
# df_model = m[m['order_status'] == 'delivered'].copy()

# # Creamos la columna 'is_delayed' (1 si se retrasó, 0 si llegó a tiempo)
# df_model['is_delayed'] = (df_model['order_delivered_customer_date'] > 
#                           df_model['order_estimated_delivery_date']).astype(int)

# print(f"Distribución de retrasos:\n{df_model['is_delayed'].value_counts(normalize=True) * 100}")